# RLT-RSI Experiment — Colab Walkthrough

Runs the repository's **actual** code on a fresh Colab runtime, in two stages:

- **Stage A — CPU/NumPy smoke (always runs).** Baseline and RSI-style NumPy adaptation.
- **Stage B — Torch GPU (if CUDA is available).** A small end-to-end Torch baseline plus
  bounded RSI-style iterative adaptation.

Terminology is deliberate: this is **RSI-style iterative adaptation** over a bounded loop
schedule, not unrestricted recursive self-improvement. The held-out split is used only for
the final post-selection evaluation.

All outputs go to temporary directories. The committed `results/` artifacts are never
overwritten.


## 1. Runtime check

In [ ]:
import platform
import sys

print("python:", sys.version.split()[0])
print("platform:", platform.platform())

import numpy as np
print("numpy:", np.__version__)

try:
    import torch
    print("torch:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu:", torch.cuda.get_device_name(0))
except ImportError:
    print("torch: not installed (Stage B will be skipped)")


## 2. Clone and install the repository

Clones `main`, installs the package, and imports the real `rlt_rsi` modules. Re-running is
safe: an existing clone is reused.


In [ ]:
import os
import subprocess
import sys

REPO_DIR = "rlt-rsi-experiment"

if not os.path.isdir(REPO_DIR):
    subprocess.check_call([
        "git", "clone", "--depth", "1",
        "https://github.com/rustfuture/rlt-rsi-experiment.git", REPO_DIR,
    ])

os.chdir(REPO_DIR)
sys.path.insert(0, os.getcwd())

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])

import rlt_rsi

print("working directory:", os.getcwd())
print("imported rlt_rsi from:", os.path.dirname(rlt_rsi.__file__))


## 3. Stage A — NumPy baseline smoke (CPU)

Frozen-feature NumPy path: this validates plumbing, determinism, and aggregation. It is
**not** evidence about whether trained recurrence helps.


In [ ]:
import json
import tempfile

stage_a_dir = tempfile.mkdtemp(prefix="rlt_stage_a_")
smoke_path = os.path.join(stage_a_dir, "numpy_smoke.json")

subprocess.check_call([
    sys.executable, "-m", "rlt_rsi.train",
    "--backend", "numpy", "--seeds", "7,42,123",
    "--train-size", "64", "--dev-size", "32", "--heldout-size", "32",
    "--epochs", "4", "--skip-diagnostic",
    "--output", smoke_path,
], cwd=os.getcwd())

with open(smoke_path, encoding="utf-8") as handle:
    smoke = json.load(handle)

def stat(value):
    """Aggregate metrics are {"mean", "std"} across seeds; single-seed values are scalar."""
    return value["mean"] if isinstance(value, dict) else value


rows = []
for row in smoke["results"]:
    m = row["metrics"]
    rows.append({
        "model": row["model"],
        "loops": row["loop_count"],
        "decision": row["decision"],
        "train_acc": round(stat(m["train_accuracy"]), 4),
        "dev_acc": round(stat(m["dev_accuracy"]), 4),
        "heldout_acc": round(stat(m["heldout_accuracy"]), 4),
        "length_generalization_drop": round(stat(m["length_generalization_drop"]), 4),
        "seconds": round(stat(m["seconds"]), 3),
        "estimated_block_calls": row["estimated_block_calls"],
    })

print("backend:", smoke["backend"])
print("scope:", smoke["scope_statement"][:140], "...")
print(json.dumps(rows, indent=2))


## 4. Stage A — RSI-style NumPy adaptation (CPU)

Bounded loop-schedule search: candidates `{k-1, k, k+1}` (clamped to 1–8) are trained,
scored on `dev`, and the best is carried forward. The held-out split is read only at the
end. Each lineage entry records the post-training `train_bce`.


In [ ]:
rsi_path = os.path.join(stage_a_dir, "rsi_numpy.json")

subprocess.check_call([
    sys.executable, "-m", "rlt_rsi.train_rsi",
    "--backend", "numpy", "--seeds", "7,42,123",
    "--train-size", "64", "--dev-size", "32", "--heldout-size", "32",
    "--epochs", "4", "--rsi-generations", "2", "--rsi-epochs-per-gen", "2",
    "--skip-diagnostic",
    "--output", rsi_path,
], cwd=os.getcwd())

with open(rsi_path, encoding="utf-8") as handle:
    rsi = json.load(handle)

rsi_row = [row for row in rsi["results"] if row["model"] == "rsi_looped"][0]
for run in rsi_row["per_seed_runs"]:
    lineage = run["lineage"]
    print(f"seed {run['seed']}: final loop count {lineage[-1]['accepted_loop_count']}, "
          f"final_train_bce {run['final_train_bce']:.4f}, "
          f"train_acc {run['train']['accuracy']:.4f}, "
          f"dev_acc {run['dev']['accuracy']:.4f}, "
          f"heldout_acc {run['heldout']['accuracy']:.4f}")
    for gen in lineage:
        print(f"   gen {gen['generation']}: proposals={gen['proposals']} "
              f"accepted={gen['accepted_loop_count']} "
              f"dev_acc={gen['dev_accuracy']:.4f} train_bce={gen['train_bce']:.4f}")


## 5. Stage B — Torch GPU (optional)

Requires a GPU runtime (`Runtime` → `Change runtime type` → GPU). If CUDA is unavailable,
this stage is skipped with an explicit message — the NumPy Stage A results remain valid
smoke evidence.


In [ ]:
torch_available = False
cuda_available = False

try:
    import torch
    torch_available = True
    cuda_available = torch.cuda.is_available()
except ImportError:
    torch = None

if cuda_available:
    print("Stage B enabled on:", torch.cuda.get_device_name(0))
elif torch_available:
    print("Stage B skipped: torch is installed but no CUDA GPU is available.")
else:
    print("Stage B skipped: torch is not installed.")


## 6. Stage B — torch baseline and RSI-style adaptation

Small, bounded sizes (single seed) so the run fits a free GPU session. `--device cuda`
fails loudly if CUDA is unavailable; there is no silent fallback.


In [ ]:
if cuda_available:
    stage_b_dir = tempfile.mkdtemp(prefix="rlt_stage_b_")
    baseline_path = os.path.join(stage_b_dir, "torch_baseline.json")
    torch_rsi_path = os.path.join(stage_b_dir, "torch_rsi.json")

    subprocess.check_call([
        sys.executable, "-m", "rlt_rsi.train",
        "--backend", "torch", "--device", "cuda", "--seeds", "7",
        "--train-size", "128", "--dev-size", "64", "--heldout-size", "64",
        "--epochs", "10", "--skip-diagnostic",
        "--output", baseline_path,
    ], cwd=os.getcwd())

    subprocess.check_call([
        sys.executable, "-m", "rlt_rsi.train_rsi",
        "--backend", "torch", "--device", "cuda", "--seeds", "7",
        "--train-size", "128", "--dev-size", "64", "--heldout-size", "64",
        "--epochs", "10", "--rsi-generations", "2", "--rsi-epochs-per-gen", "2",
        "--skip-diagnostic",
        "--output", torch_rsi_path,
    ], cwd=os.getcwd())

    with open(baseline_path, encoding="utf-8") as handle:
        baseline = json.load(handle)
    print("== torch baseline / looped ==")
    for row in baseline["results"]:
        m = row["metrics"]
        print(f"  {row['model']} loops={row['loop_count']} "
              f"dev={m['dev_accuracy']:.4f} heldout={m['heldout_accuracy']:.4f} "
              f"seconds={m['seconds']:.2f}")

    with open(torch_rsi_path, encoding="utf-8") as handle:
        torch_rsi = json.load(handle)
    torch_rsi_row = [row for row in torch_rsi["results"] if row["model"] == "rsi_looped"][0]
    torch_run = torch_rsi_row["per_seed_runs"][0]
    print("== torch RSI lineage ==")
    for gen in torch_run["lineage"]:
        print(f"  gen {gen['generation']}: proposals={gen['proposals']} "
              f"accepted={gen['accepted_loop_count']} "
              f"dev_acc={gen['dev_accuracy']:.4f} train_bce={gen['train_bce']:.4f}")
    print(f"  final_train_bce={torch_run['final_train_bce']:.4f} "
          f"heldout_acc={torch_run['heldout']['accuracy']:.4f} "
          f"seconds={torch_run['seconds']:.2f}")
    print("  parameter_counts:", torch_rsi_row["parameter_counts"])
    print("  sequential_block_applications:", torch_rsi_row["sequential_block_applications"])
else:
    print("Stage B was not run on this runtime.")


## 7. Artifacts and interpretation

- **Stage A / NumPy** used the frozen-feature smoke path. It verifies plumbing,
  determinism, and aggregation; it does not measure whether trained recurrence helps.
- **Stage B / Torch** trains end-to-end on a single seed with small sample sizes. The
  ±0.05 decision rule is an operational rule, not a significance test.
- **Held-out** numbers are post-selection only.
- These are local demonstrations. The committed reference artifacts live in
  `results/` (for example `results/torch-cpu-dev-reference/`).

Export example (Colab only):

```python
# import shutil
# archive = shutil.make_archive("/tmp/rlt_artifacts", "zip", stage_a_dir)
# from google.colab import files
# files.download(archive)
```
